# 05 - Train the v3 Standard-KD multimodal student

Runs the exact frozen v3 Standard KD: equal-weight normalized audio/text teacher soft targets, temperature 2.0, KD weight 0.5.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, shutil, subprocess, base64

DAIC = Path('/content/drive/MyDrive/DAIC_WOZ')
EXP_ROOT = DAIC / 'experiments' / 'exact_v3_reproduction'
ACTIVE = EXP_ROOT / 'ACTIVE_RUN.txt'
assert ACTIVE.is_file(), 'Run notebooks/00_start_new_run.ipynb first.'
RUN = EXP_ROOT / ACTIVE.read_text().strip()
assert RUN.is_dir(), RUN

REPO = 'engrkhubabahmad/reliability-aware-depression-kd'
CORE_COMMIT = '29087f39a46ddf756b6a082ff7148cf90e336ca7'
CODE = Path('/content/reliability-aware-depression-kd-exact-v3')

token = userdata.get('GITHUB_TOKEN')
assert token, 'Add GITHUB_TOKEN to Colab Secrets.'
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
env = os.environ.copy()
env['GIT_CONFIG_COUNT']='1'
env['GIT_CONFIG_KEY_0']='http.https://github.com/.extraheader'
env['GIT_CONFIG_VALUE_0']=f'AUTHORIZATION: basic {auth}'

if CODE.exists(): shutil.rmtree(CODE)
CODE.mkdir(parents=True)
subprocess.run(['git','-C',str(CODE),'init'],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'remote','add','origin',f'https://github.com/{REPO}.git'],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'fetch','--depth','1','origin',CORE_COMMIT],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'checkout','--detach','FETCH_HEAD'],check=True,env=env)
head=subprocess.check_output(['git','-C',str(CODE),'rev-parse','HEAD'],text=True).strip()
assert head==CORE_COMMIT,(head,CORE_COMMIT)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements.txt')],check=True)
print('RUN:',RUN)
print('Pinned core:',head)


In [ ]:
TEXT=RUN/'03_student_text'
AUDIO=RUN/'03_student_audio'
TTEACH=RUN/'01_text_teacher'
ATEACH=RUN/'02_audio_teacher'
NOKD=RUN/'04_no_kd'
OUT=RUN/'05_standard_kd'
for p in [TEXT,AUDIO,TTEACH,ATEACH,NOKD]: assert p.is_dir(),p

subprocess.run([sys.executable,'-m','scripts.students.participant_student_v3.train_fusion_standard_kd',
 '--text-branch',str(TEXT),'--audio-branch',str(AUDIO),
 '--teacher-text',str(TTEACH),'--teacher-audio',str(ATEACH),
 '--no-kd-reference',str(NOKD),'--output',str(OUT),
 '--seed','103','--epochs','80','--patience','20',
 '--temperature','2.0','--kd-weight','0.5'],cwd=CODE,check=True)

d=json.loads((OUT/'metrics.json').read_text()); m=d['dev34']; p=d['protocol']
print(json.dumps(m,indent=2))
assert p['train_participants']==107 and p['dev_participants']==34
assert p['participant_440_excluded'] is True
assert p['threshold']==0.5 and p['threshold_search'] is False
assert p['temperature']==2.0 and p['kd_weight']==0.5
assert p['test_opened'] is False
assert round(float(m['macro_f1']),4)==0.7236
assert round(float(m['depressed_f1']),4)==0.6667
assert round(float(m['auroc']),4)==0.7668
print('PASS: Standard KD DEV-34 reproduced.')